In [4]:
import numpy as np
from calico import calibration_qa
import pyuvdata

In [5]:
cal = pyuvdata.UVCal()
cal.read("/lustre/pipeline/calibration/results/2026-04-07/11h/successful/20260407_220439/tables/calibration_2026-04-07_11h.B.flagged")

Setting telescope_location to value in known_telescopes for OVRO-LWA.
Unknown polarization basis for solutions, jones_array values may be spurious.
Unknown x_orientation basis for solutions, assuming "east".


In [13]:
freq_inds = np.digitize(cal.freq_array,cal.freq_array)

In [14]:
freq_inds

array([   1,    2,    3, ..., 3070, 3071, 3072], shape=(3072,))

In [ ]:
gain = np.ma.masked_where(cal.flag_array,cal.gain_array)
gain = np.ma.sum(gain,axis=2)
#freq_inds = np.digitize(cal.freq_array,cal.freq_array)
freq_inds = np.arange(cal.Nfreqs)
print(f'selecting {len(freq_inds)} calibration channels from the available {cal.Nfreqs}')
gain = gain[:,freq_inds,:]
gain.shape = (cal.Nants_data,len(freq_inds),1,cal.Njones)

#problem. we cant combine flags because the time staggering means that every time is flagged at least once
flags = np.zeros(gain.shape, dtype=bool)
flags[np.abs(gain)>.999] = 1   #in my file, gain=1 was flagged.
flags[np.abs(gain)<1e-9] = 1   #no gain no pain
print(flags.shape)
print(f'flagging  {float(flags.sum()/flags.size)*100:4.1f}% of channels')
print(flags.shape)
flags.dtype = bool
print(flags.shape)

UVC2 = pyuvdata.UVCal()
UVC2.read("/lustre/pipeline/calibration/results/2026-04-07/11h/successful/20260407_220439/tables/calibration_2026-04-07_11h.B.flagged")
UVC2.select(times = UVC2.time_array[-1], frequencies = cal.freq_array)
UVC2.flex_spw_id_array = np.zeros_like(cal.freq_array)
UVC2.spw_array = np.array([0])
UVC2.Nspws = 1
UVC2.gain_array = gain
print(flags.shape)
UVC2.flag_array = flags

selecting 3072 calibration channels from the available 3072
(352, 3072, 1, 2)
flagging  32.6% of channels
(352, 3072, 1, 2)
(352, 3072, 1, 16)
(352, 3072, 1, 16)


In [34]:
flags = np.zeros((352, 3072, 1, 2))
print(flags.shape)
flags.dtype = bool
print(flags.shape)

(352, 3072, 1, 2)
(352, 3072, 1, 16)


In [28]:
np.shape(UVC2.flag_array)

(352, 3072, 1, 16)

In [29]:
np.shape(flags)

(352, 3072, 1, 16)

In [22]:
calibration_qa.plot_gains(UVC2,savefig=False)

IndexError: index 6 is out of bounds for axis 3 with size 2

In [25]:
np.shape(UVC2.gain_array)

(352, 3072, 1, 2)

In [26]:
np.shape(UVC2.flag_array)

(352, 3072, 1, 16)